In [1]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
from utils.animation.processing.bvh_converter import BVHParser
import time

In [2]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation

animation_visualisation.init_visualization()


Viewer URL: http://localhost:8000/utils/animation/visualisation/new/animation_viewer.html?wsport=8700


In [7]:
import soundfile as sf

target_joints = ['body_world', 'b_root', 'b_r_foot', 'b_l_foot', 'b_l_upleg', 'b_l_leg', 'b_r_upleg', 'b_r_leg', 
                'b_spine0', 'b_spine1', 'b_spine2', 'b_spine3', 'b_l_shoulder', 'b_l_arm', 'b_l_arm_twist', 
                'b_l_forearm', 'b_l_wrist_twist', 'b_l_wrist', 'b_l_pinky1', 'b_l_pinky2', 'b_l_pinky3', 'b_l_ring1', 
                'b_l_ring2', 'b_l_ring3', 'b_l_middle1', 'b_l_middle2', 'b_l_middle3', 'b_l_index1', 'b_l_index2', 
                'b_l_index3', 'b_l_thumb0', 'b_l_thumb1', 'b_l_thumb2', 'b_l_thumb3', 'b_r_shoulder', 'b_r_arm', 
                'b_r_arm_twist', 'b_r_forearm', 'b_r_wrist_twist', 'b_r_wrist', 'b_r_thumb0', 'b_r_thumb1', 
                'b_r_thumb2', 'b_r_thumb3', 'b_r_pinky1', 'b_r_pinky2', 'b_r_pinky3', 'b_r_middle1', 'b_r_middle2', 
                'b_r_middle3', 'b_r_ring1', 'b_r_ring2', 'b_r_ring3', 'b_r_index1', 'b_r_index2', 'b_r_index3', 
                'b_neck0', 'b_head']

bvh_file = "dataset/genea2023_dataset/toy/main-agent/bvh/trn_2023_v0_005_main-agent.bvh"

audio_file = "dataset/genea2023_dataset/toy/main-agent/wav/trn_2023_v0_005_main-agent.wav"
audio_data, samplerate = sf.read(audio_file)

# Load the BVH file and extract features
parser = BVHParser(bvh_file, target_joints)

features = torch.tensor(parser.to_features()).to(torch.float32)

# Calculate frames to stream
num_frames = features.shape[0]

# I want to calculate the world position of the joints.
world_positions = parser.skeleton.calculate_world_positions(frame_data=features)

# reshape to (batch_size, num_frames, num_joints, 3)
world_positions = world_positions.reshape(num_frames, len(parser.skeleton.target_joints), 3)

animation_visualisation.send_character("new_character", (0.0,0.0,1.5), 180, 0x7f7ff4)

duration_in_seconds = len(audio_data) / samplerate
frame_duration = 1.0 / 30.0  # Assuming 30 FPS
start_time = time.time()

audio_chunk_duration = 0.5  # 1-second chunks
overlap_duration = 0.1  # 100ms overlap
last_chunk_time = -0.1  # Start below 0 to trigger first chunk immediately

for i in range(num_frames):
    # Send animation pose
    animation_visualisation.send_pose(features[i], parser.skeleton, "default")
    animation_visualisation.send_pose(features[i+200], parser.skeleton, "new_character")
    
    # Current position in audio timeline
    current_audio_time = i * frame_duration
    
    # Check if we need to send a new audio chunk
    if current_audio_time >= last_chunk_time + (audio_chunk_duration - overlap_duration):
        # Calculate chunk positions with overlap
        start_index = int((current_audio_time - overlap_duration) * samplerate)
        start_index = max(0, start_index)  # Ensure we don't go negative
        end_index = start_index + int(audio_chunk_duration * samplerate)
        
        if start_index < len(audio_data) and end_index <= len(audio_data):
            audio_chunk = audio_data[start_index:end_index]
            animation_visualisation.send_audio(audio_chunk, samplerate)
            last_chunk_time = current_audio_time
    
    # Frame timing logic
    current_time = time.time()
    expected_time = start_time + i * frame_duration
    time_to_wait = expected_time - current_time
    if time_to_wait > 0:
        time.sleep(time_to_wait)

KeyboardInterrupt: 

In [ ]:
animation_visualisation.send_character(
    name = "Ground Truth",
    position = (-2, 0, 0),
    rotation = 0.0,
    color = 0x81e2b5
)

In [70]:
animation_visualisation.send_debug_tensor(torch.rand(32, 32), "Test Tensor 2")